# rssjobs / Jobber RSS test\n\nExplore job-search RSS feeds generated by [Jobber](https://github.com/alwedo/jobber) or [rssjobs.app](https://rssjobs.app/).  \n**Setup:** run `uv sync` (or install from `requirements.txt`), then run the cells below.\n\n*This notebook is read-only (HTTP GET only) and keeps memory small by limiting rows.*

## 1. Check environment

In [ ]:
import sys\nimport requests\nimport feedparser\nimport pandas as pd\n\nprint(f"Python: {sys.version}")\nprint(f"requests: {requests.__version__}")\nprint(f"feedparser: {feedparser.__version__}")\nprint(f"pandas: {pd.__version__}")

## 2. RSS feed URL and options

In [ ]:
import ipywidgets as w\nfrom IPython.display import display as ipy_display\n\n# Paste any rssjobs/jobber RSS URL here. Example placeholders only – replace with a real URL.\nrss_url = w.Text(\n    value="",\n    placeholder="https://rssjobs.app/your-feed-id.xml or http://localhost/feed/...",\n    description="RSS URL",\n    style={"description_width": "80px"},\n    layout=w.Layout(width="90%"),\n)\n\nmax_items = w.IntSlider(value=20, min=5, max=100, step=5, description="Max rows")\nkeyword = w.Text(value="", placeholder="Optional filter in title/summary", description="Keyword")\n\nout = w.Output()\n\ndef fetch_feed(_):\n    url = rss_url.value.strip()\n    with out:\n        out.clear_output()\n        if not url:\n            print("Please paste an RSS URL from rssjobs.app or your local Jobber instance.")\n            return\n        print(f"Fetching: {url}")\n        try:\n            resp = requests.get(url, timeout=15)\n            resp.raise_for_status()\n        except Exception as e:\n            print("Request failed:", e)\n            return\n\n        parsed = feedparser.parse(resp.content)\n        if parsed.bozo:\n            print("Warning: feedparser reported issues parsing this feed.")\n\n        entries = []\n        kw = keyword.value.strip().lower()\n        for entry in parsed.entries:\n            title = getattr(entry, "title", "");\n            summary = getattr(entry, "summary", getattr(entry, "description", ""));\n            if kw and kw not in title.lower() and kw not in summary.lower():\n                continue\n            entries.append({\n                "title": title,\n                "link": getattr(entry, "link", ""),\n                "published": getattr(entry, "published", getattr(entry, "updated", "")),\n                "summary": summary,\n                "source": getattr(getattr(entry, "source", {}), "title", ""),\n            })\n            if len(entries) >= max_items.value:\n                break\n\n        if not entries:\n            print("No entries found (maybe keyword is too strict or feed is empty).")\n            return\n\n        global rss_df\n        rss_df = pd.DataFrame(entries)\n        print(f"Loaded {len(rss_df)} entries")\n        ipy_display(rss_df.head())\n\nrun_btn = w.Button(description="Fetch RSS", button_style="primary")\nrun_btn.on_click(fetch_feed)\n\nform = w.VBox([rss_url, w.HBox([max_items, keyword]), run_btn, out])\nipy_display(form)

## 3. Inspect `rss_df` (optional)

In [ ]:
try:\n    rss_df\nexcept NameError:\n    print("rss_df is not defined yet. Run the 'Fetch RSS' cell first.")\nelse:\n    print(rss_df.info())\n    display(rss_df.head())

## 4. Export to CSV (optional)

In [ ]:
import csv\n\ntry:\n    rss_df\nexcept NameError:\n    print("rss_df is not defined yet. Run the 'Fetch RSS' cell first.")\nelse:\n    out_path = "rssjobs_feed.csv"\n    rss_df.to_csv(out_path, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\", index=False)\n    print(f"Saved to {out_path}")\n    # For Marimo later: you can free memory after export with `del rss_df`.